<a href="https://colab.research.google.com/github/ealmeida04/logica-programacao/blob/main/tratamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pyspark

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Leitura e Escrita YouTube") \
    .getOrCreate()

**Ler o arquivo videos-stats.csv**

In [7]:
df_videos = spark.read.csv(
    "videos-stats.csv",
    header=True
)

**Visualizando os primeiros 8 registros**

In [8]:
df_videos.show(8)

+---+--------------------+-----------+------------+-------+--------+--------+---------+
|_c0|               Title|   Video ID|Published At|Keyword|   Likes|Comments|    Views|
+---+--------------------+-----------+------------+-------+--------+--------+---------+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech|  3407.0|   672.0| 135612.0|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech| 76779.0|  4306.0|1758063.0|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech| 63825.0|  3338.0|1564007.0|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech| 71566.0|  1426.0| 922918.0|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech| 96513.0|  5155.0|1855644.0|
|  5|Brewmaster Answer...|18fwz9Itbvo|  2021-11-05|   tech| 33570.0|  1643.0| 943119.0|
|  6|Tech Monopolies: ...|jXf04bhcjbg|  2022-06-13|   tech|135047.0|  9367.0|5937790.0|
|  7|I bought the STRA...|2TqOmtTAMRY|  2022-08-07|   tech|216935.0| 12605.0|4782514.0|
+---+--------------------+------

**5 - Visualizar o esquema do arquivo**

In [9]:
df_videos.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: string (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: string (nullable = true)
 |-- Comments: string (nullable = true)
 |-- Views: string (nullable = true)



**6 - Ler novamente inferindo o esquema**

In [10]:
df_videos = spark.read.csv(
    "videos-stats.csv",
    header=True,
    inferSchema=True
)

In [11]:
df_videos.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: double (nullable = true)
 |-- Comments: double (nullable = true)
 |-- Views: double (nullable = true)



**7 - Salvar como PARQUET**

In [12]:
df_videos.write.mode("overwrite").parquet("videos-parquet")

**8 - Ler o arquivo parquet salvo**

In [13]:
df_videos_parquet = spark.read.parquet("videos-parquet")

**9 - Salvar como tabela Spark (catalog)**

In [14]:
df_videos_parquet.write \
    .mode("overwrite") \
    .saveAsTable("tb_videos")

**10 - Listar tabelas do catálogo**

In [15]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|tb_videos|      false|
+---------+---------+-----------+



**11 - Ler a tabela usando Spark SQL**

In [16]:
spark.sql("""
SELECT *
FROM tb_videos
LIMIT 10
""").show()

+---+--------------------+-----------+------------+-------+--------+--------+---------+
|_c0|               Title|   Video ID|Published At|Keyword|   Likes|Comments|    Views|
+---+--------------------+-----------+------------+-------+--------+--------+---------+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech|  3407.0|   672.0| 135612.0|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech| 76779.0|  4306.0|1758063.0|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech| 63825.0|  3338.0|1564007.0|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech| 71566.0|  1426.0| 922918.0|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech| 96513.0|  5155.0|1855644.0|
|  5|Brewmaster Answer...|18fwz9Itbvo|  2021-11-05|   tech| 33570.0|  1643.0| 943119.0|
|  6|Tech Monopolies: ...|jXf04bhcjbg|  2022-06-13|   tech|135047.0|  9367.0|5937790.0|
|  7|I bought the STRA...|2TqOmtTAMRY|  2022-08-07|   tech|216935.0| 12605.0|4782514.0|
|  8|15 Emerging Techn...|wLlL46

**12 - Ler o arquivo comments.csv inferindo schema**

In [17]:
df_comments = spark.read.csv(
    "comments.csv",
    header=True,
    inferSchema=True
)

In [18]:
df_comments.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes: string (nullable = true)
 |-- Sentiment: string (nullable = true)



**13 - Salvar comments como parquet**

In [19]:
df_comments.write \
    .mode("overwrite") \
    .parquet("comments-parquet")

*Obrigado, espero que seja isso. :)*

## ***tratamento.ipynb - seguindo.***

In [98]:
from pyspark.sql.functions import col, when, year
from pyspark.sql.types import IntegerType
from pyspark.sql import functions as F

df_video = spark.read.csv(
    "videos-stats.csv",
    header=True,
    inferSchema=True
)

df_video.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: double (nullable = true)
 |-- Comments: double (nullable = true)
 |-- Views: double (nullable = true)



**2) Substituir nulos em Likes, Comments e Views por 0**

In [99]:
df_video = df_video.fillna({
    "Likes": 0,
    "Comments": 0,
    "Views": 0
})

**3) Ler comments.csv**

In [100]:
df_comentario = spark.read.csv(
    "comments.csv",
    header=True,
    inferSchema=True
)

df_comentario.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes: string (nullable = true)
 |-- Sentiment: string (nullable = true)



**4) Contagem inicial**

In [101]:
print("Videos:", df_video.count())
print("Comentarios:", df_comentario.count())

Videos: 1881
Comentarios: 30036


**5) Remover Video ID nulos**

In [102]:
df_video = df_video.filter(col("Video ID").isNotNull())
df_comentario = df_comentario.filter(col("Video ID").isNotNull())

print("Videos após limpeza:", df_video.count())
print("Comentarios após limpeza:", df_comentario.count())

Videos após limpeza: 1881
Comentarios após limpeza: 22555


**6) Remover duplicados no df_video**

In [103]:
df_video = df_video.dropDuplicates(["Video ID"])

**7) Converter tipos corretamente**

In [119]:
from pyspark.sql.types import IntegerType, LongType
from pyspark.sql.functions import col

df_video = df_video.withColumn(
    "Likes",
    col("Likes").cast("double").cast(IntegerType())
)

df_video = df_video.withColumn(
    "Comments",
    col("Comments").cast("double").cast(IntegerType())
)

df_video = df_video.withColumn(
    "Views",
    col("Views").cast("double").cast(LongType())   # <-- ESSENCIAL
)

**8) Ajustar df_comentario**

In [122]:
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import col

df_comentario = df_comentario.withColumn(
    "Likes Comment",
    col("Likes Comment").cast("double").cast(IntegerType())
)

df_comentario = df_comentario.withColumn(
    "Sentiment",
    col("Sentiment").cast("double").cast(IntegerType())
)

In [126]:
df_comentario.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)



In [127]:
df_video.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: long (nullable = true)
 |-- Year: integer (nullable = true)



In [128]:
df_video.count()

1869

**9) Criar Interaction**

In [121]:
df_video = df_video.withColumn(
    "Interaction",
    (col("Likes") + col("Comments") + col("Views")).cast(LongType())
)

**10) Converter Published At para date**

In [114]:
df_video = df_video.withColumn(
    "Published At",
    F.to_date(col("Published At"))
)

**11) Criar Year**

In [115]:
df_video = df_video.withColumn(
    "Year",
    year(col("Published At"))
)

**12) Join comentários**

In [130]:
df_video.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: long (nullable = true)
 |-- Year: integer (nullable = true)



In [133]:
spark.catalog.clearCache()

In [134]:
df_video = spark.read.csv(
    "videos-stats.csv",
    header=True,
    inferSchema=True
)

In [135]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, LongType

df_video = df_video.fillna({
    "Likes": 0,
    "Comments": 0,
    "Views": 0
})

df_video = df_video.withColumn("Likes", col("Likes").cast(IntegerType()))
df_video = df_video.withColumn("Comments", col("Comments").cast(IntegerType()))
df_video = df_video.withColumn("Views", col("Views").cast(LongType()))

df_video = df_video.withColumn(
    "Interaction",
    (col("Likes") + col("Comments") + col("Views")).cast(LongType())
)

In [136]:
df_video.count()

1881

In [137]:
df_video.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: long (nullable = true)



In [138]:
df_join_video_comments = df_video.join(
    df_comentario,
    on="Video ID",
    how="left"
)

In [139]:
df_join_video_comments.count()

18645

**13) Ler USvideos.csv**

In [140]:
df_us_videos = spark.read.csv(
    "USvideos.csv",
    header=True,
    inferSchema=True
)

In [141]:
df_us_videos.count()

48137

**14) Join por Title**

In [142]:
df_join_video_usvideos = df_video.join(
    df_us_videos,
    on="Title",
    how="left"
)

In [143]:
df_join_video_usvideos.count()

1896

In [144]:
df_join_video_usvideos = df_video.join(
    df_us_videos,
    on="Title",
    how="left"
)

In [145]:
df_join_video_usvideos.count()

1896

**15 - Verifique a quantidade de campos nulos em todos os campos do dataframe df_video**

In [146]:
from pyspark.sql.functions import col, count, when

df_video.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_video.columns
]).show()

+---+-----+--------+------------+-------+-----+--------+-----+-----------+
|_c0|Title|Video ID|Published At|Keyword|Likes|Comments|Views|Interaction|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+
|  0|    0|       0|           0|      0|    0|       0|    0|          0|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+



**16 - Remova a coluna '_c0' e salve o dataframe df_video como 'videos-tratados-parquet' no formato parquet e adicione o cabeçalho nos dados**

In [147]:
df_video = df_video.drop("_c0")

In [148]:
df_video.printSchema()

root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: long (nullable = true)



In [150]:
df_video.write.mode("overwrite").parquet("videos-tratados-parquet")

In [151]:
df_validacao = spark.read.parquet("videos-tratados-parquet")
df_validacao.count()

1881

**17 - Remova a coluna '_c0' e salve o dataframe df_join_video_comments como 'videos-comments-tratados-parquet' no formato parquet e adicione o cabeçalho nos dados**

In [152]:
df_join_video_comments.printSchema()

root
 |-- Video ID: string (nullable = true)
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: long (nullable = true)
 |-- _c0: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)



In [178]:
df_join_video_comments = df_join_video_comments.drop("_c0")

In [186]:
df_comentario.columns

['_c0', 'Video ID', 'Comment', 'Sentiment', 'Likes Comment']

In [187]:
df_comentario.select("Likes Comment", "Sentiment").show(10)

+-------------+---------+
|Likes Comment|Sentiment|
+-------------+---------+
|           95|        1|
|           19|        0|
|          161|        2|
|            8|        0|
|           34|        2|
|            8|        1|
|           29|        2|
|            7|        1|
|            2|        2|
|           28|        1|
+-------------+---------+
only showing top 10 rows


In [182]:
df_join_video_comments = df_join_video_comments.drop("_c0")

In [192]:
from pyspark.sql.functions import col, when, year, to_date, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType, DoubleType
import pyspark.sql.functions as F

# Define schema for comments.csv explicitly to avoid inference issues
comments_schema = StructType([
    StructField("_c0", StringType(), True),
    StructField("Video ID", StringType(), True),
    StructField("Comment", StringType(), True),
    StructField("Likes", StringType(), True), # Keep as string initially
    StructField("Sentiment", StringType(), True) # Keep as string initially
])

# Re-initialize df_video (similar to start of 'tratamento.ipynb')
df_video = spark.read.csv(
    "videos-stats.csv",
    header=True,
    inferSchema=True
)
df_video = df_video.fillna({
    "Likes": 0,
    "Comments": 0,
    "Views": 0
})
df_video = df_video.filter(col("Video ID").isNotNull())
df_video = df_video.dropDuplicates(["Video ID"])
df_video = df_video.withColumn("Likes", col("Likes").cast("double").cast(IntegerType()))
df_video = df_video.withColumn("Comments", col("Comments").cast("double").cast(IntegerType()))
df_video = df_video.withColumn("Views", col("Views").cast("double").cast(LongType()))
df_video = df_video.withColumn(
    "Interaction",
    (col("Likes") + col("Comments") + col("Views")).cast(LongType())
)
df_video = df_video.withColumn(
    "Published At",
    F.to_date(col("Published At"))
)
df_video = df_video.withColumn(
    "Year",
    year(col("Published At"))
)
# Drop original _c0 from df_video as it's not needed
df_video = df_video.drop("_c0")


# Re-initialize df_comentario with explicit schema
df_comentario = spark.read.csv(
    "comments.csv",
    header=True,
    schema=comments_schema # Use explicit schema here
)
df_comentario = df_comentario.filter(col("Video ID").isNotNull())
df_comentario = df_comentario.withColumnRenamed("_c0", "comment_original_index") # Renaming to avoid _c0 ambiguity
df_comentario = df_comentario.withColumn(
    "Likes Comment",
    F.expr("try_cast(Likes as double)").cast("int")
)
df_comentario = df_comentario.withColumn(
    "Sentiment",
    F.expr("try_cast(Sentiment as double)").cast("int")
)
df_comentario = df_comentario.drop("Likes") # Drop original Likes string column


# Perform the join
df_join_video_comments = df_video.join(
    df_comentario,
    on="Video ID",
    how="left"
)

# Select and cast all columns explicitly for final write
df_final_to_write = df_join_video_comments.select(
    col("Video ID").cast(StringType()).alias("Video ID"),
    col("Title").cast(StringType()).alias("Title"),
    col("Published At").cast(DateType()).alias("Published At"),
    col("Keyword").cast(StringType()).alias("Keyword"),
    col("Likes").cast(IntegerType()).alias("Likes"),
    col("Comments").cast(IntegerType()).alias("Comments"),
    col("Views").cast(LongType()).alias("Views"),
    col("Interaction").cast(LongType()).alias("Interaction"),
    col("Year").cast(IntegerType()).alias("Year"),
    col("comment_original_index").cast(StringType()).alias("comment_original_index"),
    col("Comment").cast(StringType()).alias("Comment"),
    col("Sentiment").cast(IntegerType()).alias("Sentiment"),
    col("Likes Comment").cast(IntegerType()).alias("Likes Comment")
)

df_final_to_write.write \
    .mode("overwrite") \
    .parquet("videos-comments-tratados-parquet")